# Practice Lab: Observability — Langfuse & LangSmith (Lesson 7.6)

Module 7 · 8 exercises · Tracing + scoring + dashboards + compliance

Exercises 1, 2, 3, 4, 6, 8 run locally.


# Lesson 7.6: Observability — Langfuse & LangSmith

8 exercises: 2E + 3M + 3C

Exercises 1, 2, 3, 4, 6, 8 run **locally**. Ex 5, 7 are architecture/design.


In [ ]:
import time, random, uuid, re, statistics, json
from collections import defaultdict


---
## Exercise 1 (Easy): Langfuse @observe Tracing


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import time, random, uuid

class SimTrace:
    def __init__(self, name, user_id="anon"):
        self.id = str(uuid.uuid4())[:8]
        self.name = name
        self.user_id = user_id
        self.spans = []
        self.start = time.time()
    def span(self, name, stype="span"):
        s = {"name": name, "type": stype, "start": time.time()}
        self.spans.append(s)
        return s
    def end_span(self, s, tokens=None, cost=0):
        s["latency_ms"] = round((time.time() - s["start"]) * 1000)
        s["tokens"] = tokens or {}
        s["cost"] = cost
    def report(self):
        total_ms = round((time.time() - self.start) * 1000)
        total_cost = sum(s["cost"] for s in self.spans)
        print(f"\nTrace: {self.name} ({self.id})")
        print(f"  User: {self.user_id}, Total: {total_ms}ms, ${total_cost:.6f}")
        for s in self.spans:
            icon = "GEN" if s["type"] == "generation" else "SPAN"
            tok = f" [{s['tokens'].get('input',0)}+{s['tokens'].get('output',0)} tok]" if s['tokens'] else ""
            print(f"    [{icon:>4}] {s['name']:<20} {s['latency_ms']:>5}ms ${s['cost']:.6f}{tok}")

traces = []
for q, uid in [("What is RAG?", "student_001"), ("Explain embeddings", "student_002"), ("How does retrieval work?", "student_001")]:
    t = SimTrace("rag_pipeline", uid)
    s1 = t.span("embed_question")
    time.sleep(random.uniform(0.01, 0.03))
    t.end_span(s1, tokens={"input": len(q.split())})
    s2 = t.span("retrieve_docs")
    time.sleep(random.uniform(0.01, 0.04))
    t.end_span(s2)
    s3 = t.span("generate_response", stype="generation")
    time.sleep(random.uniform(0.03, 0.08))
    in_t, out_t = random.randint(100, 300), random.randint(50, 150)
    cost = (in_t * 0.075 + out_t * 0.30) / 1e6
    t.end_span(s3, tokens={"input": in_t, "output": out_t}, cost=cost)
    t.report()
    traces.append(t)

tc = sum(sum(s["cost"] for s in t.spans) for t in traces)
print(f"\nSummary: {len(traces)} traces, ${tc:.6f} total")
print(f"Manual: trace.generation(name=..., model=...)")
print(f"Decorator: @observe(as_type='generation') -> same result")


</details>


---
## Exercise 2 (Easy): Helicone 1-Line Integration


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import random
from collections import defaultdict

PRICING = {"gpt-4o-mini": (0.15, 0.60), "gemini-2.0-flash": (0.075, 0.30)}
calls = []
for _ in range(50):
    model = random.choice(list(PRICING.keys()))
    feature = random.choice(["chat", "search", "summarize", "translate"])
    env = random.choice(["production", "staging"])
    in_t, out_t = random.randint(50, 500), random.randint(20, 200)
    p = PRICING[model]
    calls.append({"model": model, "feature": feature, "env": env,
                  "cost": (in_t*p[0] + out_t*p[1]) / 1e6, "latency": random.randint(200, 3000)})

print("Helicone: base_url='https://oai.helicone.ai/v1' + 1 header")
print(f"\nCost by Feature ({len(calls)} calls):")
by_f = defaultdict(lambda: {"cost": 0, "n": 0})
for c in calls:
    by_f[c["feature"]]["cost"] += c["cost"]
    by_f[c["feature"]]["n"] += 1
for f, d in sorted(by_f.items(), key=lambda x: -x[1]["cost"]):
    print(f"  {f:<14} {d['n']:>3} calls  ${d['cost']:.5f}")

print(f"\nCost by Model:")
by_m = defaultdict(lambda: {"cost": 0, "n": 0})
for c in calls:
    by_m[c["model"]]["cost"] += c["cost"]
    by_m[c["model"]]["n"] += 1
for m, d in by_m.items():
    print(f"  {m:<22} {d['n']:>3} calls  ${d['cost']:.5f}")

total = sum(c["cost"] for c in calls)
print(f"\nTotal: ${total:.5f}, Monthly: ${total/50*1000*30:.2f}")
print(f"Best: Helicone (cost) + Langfuse (traces) together")


</details>


---
## Exercise 3 (Medium): Prompt Management + A/B Testing


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import random

class PromptManager:
    def __init__(self):
        self.versions = {}
        self.labels = {}
    def create(self, template, temp=0.7):
        v = len(self.versions) + 1
        self.versions[v] = {"template": template, "temp": temp, "labels": []}
        return v
    def label(self, v, label):
        self.versions[v]["labels"].append(label)
        self.labels[label] = v
    def get(self, label):
        return self.versions.get(self.labels.get(label))

pm = PromptManager()
pm.create("Answer: {{question}}", 0.7)
pm.create("You are a Netsetos tutor. Answer concisely.\n\nQ: {{question}}", 0.5)
pm.create("Expert GenAI tutor. Answer in 3 bullets with code.\n\nQ: {{question}}", 0.3)
pm.label(1, "production"); pm.label(2, "staging"); pm.label(3, "experimental")

dataset = [f"Question {i+1}" for i in range(20)]
print("Prompt A/B Testing:")
results = {}
for v in [1, 2, 3]:
    scores = [min(1.0, max(0.0, 0.5 + (v-1)*0.1 + random.uniform(-0.15, 0.15))) for _ in dataset]
    avg = sum(scores)/len(scores)
    low = sum(1 for s in scores if s < 0.6)
    results[v] = avg
    print(f"  v{v} (temp={pm.versions[v]['temp']}): avg={avg:.3f}, low={low}/20")
    print(f"    {pm.versions[v]['template'][:50]}...")

winner = max(results, key=results.get)
pm.label(winner, "production")
print(f"\nWinner: v{winner} (avg={results[winner]:.3f}) -> promoted to production")


</details>


---
## Exercise 4 (Medium): RAGAS Evaluation Pipeline


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class RAGASEval:
    @staticmethod
    def faithfulness(answer, context):
        claims = [c.strip() for c in answer.split(".") if len(c.strip()) > 10]
        if not claims: return 1.0, []
        supported = 0
        details = []
        for claim in claims:
            words = set(claim.lower().split())
            ctx_words = set(context.lower().split())
            overlap = len(words & ctx_words) / max(len(words), 1)
            ok = overlap > 0.3
            supported += int(ok)
            details.append((claim[:40], ok))
        return supported / len(claims), details

    @staticmethod
    def ctx_precision(retrieved, relevant):
        if not retrieved: return 0.0
        return sum(1 for d in retrieved if d in relevant) / len(retrieved)

    @staticmethod
    def ctx_recall(retrieved, all_relevant):
        if not all_relevant: return 1.0
        return sum(1 for d in all_relevant if d in retrieved) / len(all_relevant)

cases = [
    {"q": "What is RAG?",
     "ctx": "RAG stands for Retrieval-Augmented Generation. It combines search with LLM generation.",
     "ans": "RAG stands for Retrieval-Augmented Generation. It retrieves documents and uses them for LLM generation.",
     "label": "Faithful"},
    {"q": "What is RAG?",
     "ctx": "RAG combines retrieval with generation.",
     "ans": "RAG was invented by Facebook in 2020. It achieves 95% accuracy on all benchmarks.",
     "label": "Hallucinated"},
    {"q": "What are embeddings?",
     "ctx": "Embeddings are dense vector representations capturing semantic meaning. Dimensions: 768 or 1536.",
     "ans": "Embeddings are dense vectors capturing semantic meaning with 768 or 1536 dimensions.",
     "label": "Faithful"},
    {"q": "What is fine-tuning?",
     "ctx": "Fine-tuning adjusts model weights on domain-specific data.",
     "ans": "Fine-tuning adjusts weights. It always requires 10,000 examples and costs $50,000.",
     "label": "Partially hallucinated"},
    {"q": "What is MCP?",
     "ctx": "MCP is the Model Context Protocol for AI tool integration.",
     "ans": "MCP is the Model Context Protocol created by Anthropic for standardizing AI tool integration.",
     "label": "Mostly faithful"},
]

ev = RAGASEval()
THRESHOLD = 0.7
print("RAGAS Faithfulness:")
print("=" * 55)
all_scores = []
for c in cases:
    score, details = ev.faithfulness(c["ans"], c["ctx"])
    all_scores.append(score)
    status = "PASS" if score >= THRESHOLD else "FAIL"
    print(f"  [{status}] {c['q']}: {score:.2f} ({c['label']})")
    for claim, ok in details:
        print(f"    [{'OK' if ok else '!!'}] {claim}...")

print(f"\nContext Metrics:")
print(f"  Precision: {ev.ctx_precision(['d1','d2','d3'], ['d1','d3']):.2f}")
print(f"  Recall: {ev.ctx_recall(['d1','d2','d3'], ['d1','d3','d5']):.2f}")
passed = sum(1 for s in all_scores if s >= THRESHOLD)
print(f"\nSummary: {passed}/{len(cases)} passed, avg={sum(all_scores)/len(all_scores):.2f}")


</details>


---
## Exercise 6 (Challenge): OpenTelemetry + Grafana Dashboard


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import random, statistics
from collections import defaultdict

class Dashboard:
    def __init__(self):
        self.traces = []
    def record(self, t):
        self.traces.append(t)
    def report(self):
        n = len(self.traces)
        errors = sum(1 for t in self.traces if t.get("error"))
        lats = sorted(t["latency_ms"] for t in self.traces)
        costs = [t["cost_usd"] for t in self.traces]
        scores = [t["quality"] for t in self.traces if "quality" in t]
        print(f"\nDashboard ({n} traces):")
        print(f"  Error rate: {errors/n:.1%}")
        print(f"  Latency: P50={lats[n//2]}ms P95={lats[int(n*0.95)]}ms P99={lats[int(n*0.99)]}ms")
        print(f"  Cost: ${sum(costs):.4f} total, ${statistics.mean(costs):.5f} avg, ${sum(costs)/n*1000*30:.2f}/mo")
        if scores:
            low = sum(1 for s in scores if s < 0.6)
            print(f"  Quality: {statistics.mean(scores):.2f} avg, {low} low ({low/len(scores):.0%})")
        by_m = defaultdict(lambda: {"n": 0, "cost": 0, "lat": []})
        for t in self.traces:
            by_m[t["model"]]["n"] += 1
            by_m[t["model"]]["cost"] += t["cost_usd"]
            by_m[t["model"]]["lat"].append(t["latency_ms"])
        print(f"  By Model:")
        for m, d in by_m.items():
            print(f"    {m:<22} {d['n']:>3} calls ${d['cost']:.4f} {sum(d['lat'])//len(d['lat'])}ms")
        # SLO alerts
        alerts = []
        if errors/n > 0.05: alerts.append(f"Error {errors/n:.1%}>5%")
        if lats[int(n*0.95)] > 3000: alerts.append(f"P95 {lats[int(n*0.95)]}ms>3s")
        if sum(costs) > 0.05: alerts.append(f"Cost ${sum(costs):.4f}>$0.05")
        if scores and statistics.mean(scores) < 0.7: alerts.append(f"Quality {statistics.mean(scores):.2f}<0.7")
        print(f"  Alerts: {alerts if alerts else 'All SLOs OK'}")

dash = Dashboard()
P = {"gemini-2.0-flash": (0.075, 0.30), "gpt-4o-mini": (0.15, 0.60)}
for _ in range(100):
    m = random.choice(["gemini-2.0-flash"]*7 + ["gpt-4o-mini"]*3)
    p = P[m]
    it, ot = random.randint(50, 500), random.randint(20, 200)
    dash.record({"model": m, "latency_ms": random.randint(200, 4000),
                 "cost_usd": (it*p[0]+ot*p[1])/1e6, "quality": random.uniform(0.5, 1.0),
                 "error": random.random() < 0.03})
dash.report()


</details>


---
## Exercise 8 (Challenge): India-Compliant Observability


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import re

class PIIRedactor:
    PATTERNS = [
        ("aadhaar", r'\b\d{4}\s?\d{4}\s?\d{4}\b', "[AADHAAR]"),
        ("phone", r'\b(?:\+91|91|0)?[6-9]\d{9}\b', "[PHONE]"),
        ("pan", r'\b[A-Z]{5}\d{4}[A-Z]\b', "[PAN]"),
        ("email", r'\b[\w.-]+@[\w.-]+\.\w+\b', "[EMAIL]"),
    ]
    @classmethod
    def redact(cls, text):
        out, found = text, []
        for name, pat, repl in cls.PATTERNS:
            matches = re.findall(pat, out)
            if matches:
                found.append((name, len(matches)))
                out = re.sub(pat, repl, out)
        return out, found

tests = [
    "Aadhaar 1234 5678 9012 phone +919876543210",
    "PAN: ABCDE1234F email: rahul@netsetos.com",
    "No PII here, just RAG and embeddings",
]
print("PII Redaction:")
for t in tests:
    r, f = PIIRedactor.redact(t)
    print(f"  [{'REDACTED' if f else 'CLEAN':>8}] {t[:45]}...")
    if f: print(f"    Found: {f}  Clean: {r[:50]}...")

USD_INR, GST = 84, 0.18
print(f"\nINR Cost (with 18% GST):")
print(f"{'Provider':<20} {'USD/MTok':<10} {'INR+GST':<12} {'10M tok/mo'}")
for name, usd in [("GPT-4o", 2.50), ("GPT-4o-mini", 0.15), ("Gemini Flash", 0.075), ("Sarvam 105B", 0.00)]:
    inr = usd * USD_INR * (1+GST) if usd > 0 else 0
    print(f"  {name:<18} ${usd:<8.3f} Rs {inr:<10.2f} Rs {inr*10:>8,.0f}")

print(f"\nDPDP Requirements:")
for i, r in enumerate(["Self-host AWS Mumbai", "PII redaction before ingestion",
    "User erasure capability", "Data retention policies", "Consent collection",
    "Deadline: May 13, 2027", "Penalty: Rs 250 Cr"], 1):
    print(f"  [{i}] {r}")
print(f"\nIndic eval: ChrF (not BLEU/ROUGE) for Hindi")


</details>


---
## Exercise 5 (Medium): DeepEval CI/CD Quality Gates
Architecture/design. See practice-lab-7_6.html.


In [ ]:
# Exercise 5: DeepEval CI/CD Quality Gates\npass


---
## Exercise 7 (Challenge): Self-Host Langfuse v3
Architecture/design. See practice-lab-7_6.html.


In [ ]:
# Exercise 7: Self-Host Langfuse v3\npass
